# SP500 Executive Role Classification — Fine-Tune gpt-oss-20b

Based on the [official Unsloth gpt-oss notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-Fine-tuning.ipynb), adapted for SP500 executive classification data.

**Runtime**: A100 80GB recommended. Go to Runtime → Change runtime type → A100.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    full_finetuning = False,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

## Load & Format Training Data

In [ ]:
# Clone repo with training data
REPO_DIR = "/content/finetune_sp500"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/daalbert981/finetune_sp500.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

In [ ]:
import json
from datasets import Dataset

DATA_PATH = os.path.join(REPO_DIR, "Training Datasets", "full_data_n_4977.jsonl")
with open(DATA_PATH, "r") as f:
    examples = [json.loads(line) for line in f]

print(f"Loaded {len(examples)} examples")

# Shuffle and split 95/5
import random
random.seed(42)
random.shuffle(examples)
split_idx = int(len(examples) * 0.95)

train_dataset = Dataset.from_list(examples[:split_idx])
valid_dataset = Dataset.from_list(examples[split_idx:])
print(f"Train: {len(train_dataset)}, Valid: {len(valid_dataset)}")

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

# Our data is already in OpenAI format (role/content) — no need for standardize_sharegpt
train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
valid_dataset = valid_dataset.map(formatting_prompts_func, batched = True)

print(f"Train samples: {len(train_dataset)}, Valid samples: {len(valid_dataset)}")
# Show FULL formatted text of first example to see exact tags
print("=== FULL FORMATTED TEXT ===")
print(train_dataset[0]["text"])

## Train

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2,
        num_train_epochs = 3,
        warmup_ratio = 0.03,
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "no",
        save_strategy = "steps",
        save_steps = 200,
        save_total_limit = 2,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
        bf16 = True,
        max_seq_length = 2048,
        dataloader_num_workers = 4,
        dataloader_pin_memory = True,
    ),
)

print(f"Effective batch size: {8 * 2}")
print(f"Train samples: {len(train_dataset)}")
print(f"Approx training steps: ~{len(train_dataset) * 3 // 16}")

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")

In [ ]:
# Eval during training was disabled due to an Unsloth gpt-oss attention bug.
# Skip this cell — the holdout evaluation below tests the model properly.
print("Skipping trainer.evaluate() — see holdout evaluation below.")

## Quick Inference Test

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": train_dataset[0]["messages"][0]["content"]},
    {"role": "user", "content": "In 2020 the company 'apple inc' had an executive with the name jane doe, whose official role title was: 'senior vice president, human resources'."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 200, streamer = TextStreamer(tokenizer))

## Save to HuggingFace

In [ ]:
HF_TOKEN = ""  # paste your HF write token here

# Save locally
model.save_pretrained("sp500_exec_lora")
tokenizer.save_pretrained("sp500_exec_lora")
print("LoRA adapters saved locally.")

In [ ]:
# Push merged 16-bit to HuggingFace
model.push_to_hub_merged(
    "daresearch/sp500-exec-classifier",
    tokenizer,
    save_method = "merged_16bit",
    token = HF_TOKEN,
)
print("Model pushed to https://huggingface.co/daresearch/sp500-exec-classifier")

## Holdout Evaluation

Runs predictions on all 2,010 holdout examples and computes per-label metrics.

In [ ]:
import csv, re, time

FULL_SYSTEM_PROMPT = train_dataset[0]["messages"][0]["content"]

RANK_LABELS = ["vp", "svp", "evp", "sevp", "dir", "sdir", "md", "smd", "se", "vc", "svc", "president"]
ROLE_LABELS = ["board", "ceo", "cxo", "primary", "support", "bu"]
ALL_LABELS = RANK_LABELS + ROLE_LABELS


def predict_single(role_title, company, year, name):
    user_msg = f"In {year} the company '{company}' had an executive with the name {name}, whose official role title was: '{role_title}'."
    messages = [{"role": "system", "content": FULL_SYSTEM_PROMPT}, {"role": "user", "content": user_msg}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.0, do_sample=False)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def parse_output(text):
    result = {}
    for tag in ["rank", "role"]:
        match = re.search(f"<{tag}>(.*?)</{tag}>", text)
        if match:
            for pair in match.group(1).split(";"):
                if "=" in pair:
                    k, v = pair.split("=", 1)
                    try: result[k.strip()] = int(v.strip())
                    except ValueError: result[k.strip()] = -1
    return result

print("Inference helpers loaded.")

In [ ]:
HOLDOUT_PATH = os.path.join(REPO_DIR, "Training Datasets", "holdout_labeling_09072024_template.csv")

with open(HOLDOUT_PATH, "r", encoding="latin-1") as f:
    holdout_rows = list(csv.DictReader(f))

print(f"Holdout examples: {len(holdout_rows)}")

FastLanguageModel.for_inference(model)

predictions = []
parse_failures = 0
t0 = time.time()

for i, row in enumerate(holdout_rows):
    raw = predict_single(row["role"], row["company"], int(row["year"]), row["full_name"])
    parsed = parse_output(raw)
    if not all(lbl in parsed for lbl in ALL_LABELS):
        parse_failures += 1
    predictions.append({"idx": i, "uniqueid": row["uniqueid"], "full_name": row["full_name"],
                        "company": row["company"], "role": row["role"], "year": row["year"],
                        "raw_output": raw, "parsed": parsed})
    if (i + 1) % 100 == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta = (len(holdout_rows) - i - 1) / rate
        print(f"  [{i+1}/{len(holdout_rows)}] {rate:.1f} ex/s, ETA {eta/60:.1f} min, parse failures: {parse_failures}")

print(f"\nDone! {len(predictions)} predictions in {(time.time()-t0)/60:.1f} min")
print(f"Parse failures: {parse_failures}/{len(predictions)}")

In [ ]:
import numpy as np

y_true = {lbl: [] for lbl in ALL_LABELS}
y_pred = {lbl: [] for lbl in ALL_LABELS}

for pred, row in zip(predictions, holdout_rows):
    for lbl in ALL_LABELS:
        y_true[lbl].append(int(row[lbl]))
        y_pred[lbl].append(pred["parsed"].get(lbl, -1))

print(f"{'Label':>10} | {'Acc':>6} | {'Prec':>6} | {'Recall':>6} | {'F1':>6} | {'Support':>7} | {'Parse%':>6}")
print("-" * 72)

label_metrics = {}
for lbl in ALL_LABELS:
    yt = np.array(y_true[lbl]); yp = np.array(y_pred[lbl])
    valid = (yp >= 0); parse_rate = valid.sum() / len(yp) * 100
    yt_v = yt[valid]; yp_v = yp[valid]
    if len(yt_v) == 0:
        label_metrics[lbl] = {"acc": 0, "prec": 0, "rec": 0, "f1": 0, "support": 0, "parse_rate": 0}
        print(f"{lbl:>10} | {'N/A':>6} | {'N/A':>6} | {'N/A':>6} | {'N/A':>6} | {0:>7} | {parse_rate:>5.1f}%")
        continue
    tp = ((yp_v==1)&(yt_v==1)).sum(); fp = ((yp_v==1)&(yt_v==0)).sum()
    fn = ((yp_v==0)&(yt_v==1)).sum(); tn = ((yp_v==0)&(yt_v==0)).sum()
    acc = (tp+tn)/len(yt_v)
    prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    rec = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1 = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
    support = int(yt_v.sum())
    label_metrics[lbl] = {"acc": acc, "prec": prec, "rec": rec, "f1": f1, "support": support, "parse_rate": parse_rate}
    print(f"{lbl:>10} | {acc:>5.1%} | {prec:>5.1%} | {rec:>5.1%} | {f1:>5.1%} | {support:>7} | {parse_rate:>5.1f}%")

print("-" * 72)
avg_f1 = np.mean([m["f1"] for m in label_metrics.values()])
rank_f1 = np.mean([label_metrics[l]["f1"] for l in RANK_LABELS])
role_f1 = np.mean([label_metrics[l]["f1"] for l in ROLE_LABELS])
print(f"\nMacro F1: {avg_f1:.1%}  |  Rank F1: {rank_f1:.1%}  |  Role F1: {role_f1:.1%}")

In [ ]:
# Show misclassified examples
misclassified = []
for pred, row in zip(predictions, holdout_rows):
    errors = {lbl: f"pred={pred['parsed'].get(lbl,-1)} truth={int(row[lbl])}" for lbl in ALL_LABELS if pred['parsed'].get(lbl,-1) != int(row[lbl])}
    if errors: misclassified.append((pred, row, errors))

print(f"Fully correct: {len(holdout_rows)-len(misclassified)}/{len(holdout_rows)} ({(len(holdout_rows)-len(misclassified))/len(holdout_rows):.1%})")
print(f"With errors: {len(misclassified)}/{len(holdout_rows)}\n")

for pred, row, errors in misclassified[:15]:
    print(f"{row['full_name']} - {row['role']} @ {row['company']} ({row['year']})")
    for lbl, err in errors.items(): print(f"    {lbl}: {err}")
    print()

In [ ]:
# Export results to CSV
OUTPUT_CSV = "/content/holdout_results.csv"
with open(OUTPUT_CSV, "w", newline="") as f:
    fieldnames = ["uniqueid","year","company","full_name","role","raw_output"] + \
                 [f"truth_{l}" for l in ALL_LABELS] + [f"pred_{l}" for l in ALL_LABELS] + [f"correct_{l}" for l in ALL_LABELS]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for pred, row in zip(predictions, holdout_rows):
        out = {"uniqueid": row["uniqueid"], "year": row["year"], "company": row["company"],
               "full_name": row["full_name"], "role": row["role"], "raw_output": pred["raw_output"]}
        for lbl in ALL_LABELS:
            truth = int(row[lbl]); predicted = pred["parsed"].get(lbl, -1)
            out[f"truth_{lbl}"] = truth; out[f"pred_{lbl}"] = predicted; out[f"correct_{lbl}"] = int(predicted == truth)
        writer.writerow(out)

print(f"Results exported to {OUTPUT_CSV}")
try:
    from google.colab import files; files.download(OUTPUT_CSV)
except: pass